In [ ]:
run_category_sample_demo(1)

In [ ]:
try:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = GLIDELikeModel(device=device)
            model.load_model("glide_like_model_final.pt")
            category_comparison_analysis(model)
except Exception as e:
            print(f"❌ Error: {e}")
            print("💡 Make sure you have a trained model available.")

In [ ]:
# ===================================================================
# COMPLETE RESTRUCTURED GLIDE-LIKE MODEL WITH IMPROVEMENTS
# ===================================================================

import os
import time
import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import math
import json
import warnings
from torchvision.models.inception import inception_v3
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid, save_image

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ===================================================================
# CONFIGURATION
# ===================================================================

class Config:
    # Model parameters
    BATCH_SIZE = 20  # Reduced for old systems
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 50  # Increased for better training
    SAVE_INTERVAL = 5
    IMAGE_SIZE = 64

    # Model architecture
    MODEL_CHANNELS = 256  # Reduced from 128
    TEXT_EMB_DIM = 512  # Reduced from 512
    MAX_SEQ_LEN = 50
    VOCAB_SIZE = 2000

    # Training optimizations
    USE_MIXED_PRECISION = True
    EMA_DECAY = 0.995
    GRADIENT_CLIP = 1.0
    WEIGHT_DECAY = 1e-2

    # Diffusion parameters
    NOISE_STEPS = 1000
    BETA_START = 1e-4
    BETA_END = 0.02

In [ ]:
# ===================================================================
# IMPROVED TEXT ENCODER
# ===================================================================

class ImprovedTextEncoder(nn.Module):
    def __init__(self, vocab_size=Config.VOCAB_SIZE, embed_dim=Config.TEXT_EMB_DIM,
                 max_seq_len=Config.MAX_SEQ_LEN):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        # Expanded vocabulary for restaurant domain
        self.vocab = {
            '<PAD>': 0, '<UNK>': 1, 'a': 2, 'the': 3, 'of': 4, 'photo': 5,
            'delicious': 6, 'plate': 7, 'food': 8, 'at': 9, 'restaurant': 10,
            'menu': 11, 'showing': 12, 'items': 13, 'interior': 14, 'view': 15,
            'or': 16, 'cafe': 17, 'exterior': 18, 'building': 19, 'refreshing': 20,
            'drink': 21, 'beverage': 22, 'and': 23, 'in': 24, 'with': 25,
            'dining': 26, 'table': 27, 'chair': 28, 'window': 29, 'outdoor': 30,
            'pizza': 31, 'burger': 32, 'salad': 33, 'coffee': 34, 'bar': 35,
            'kitchen': 36, 'chef': 37, 'served': 38, 'fresh': 39, 'tasty': 40,
            'cozy': 41, 'modern': 42, 'traditional': 43, 'elegant': 44, 'casual': 45,
            'meal': 46, 'dish': 47, 'served': 48, 'appetizing': 49, 'beautiful': 50
        }

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(max_seq_len, embed_dim))

        # Lighter transformer
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(embed_dim, nhead=4, batch_first=True,
                                     dim_feedforward=embed_dim*2),
            num_layers=2
        )

        self.layer_norm = nn.LayerNorm(embed_dim)

    def tokenize_caption(self, caption):
        """Tokenize a single caption"""
        words = caption.lower().replace(',', '').replace('.', '').split()
        tokens = []
        for word in words[:self.max_seq_len-1]:
            token = self.vocab.get(word, self.vocab['<UNK>'])
            tokens.append(token)

        # Pad to max_seq_len
        while len(tokens) < self.max_seq_len:
            tokens.append(self.vocab['<PAD>'])

        return tokens[:self.max_seq_len]

    def forward(self, captions):
        """Forward pass through text encoder"""
        if isinstance(captions, str):
            captions = [captions]

        tokenized = []
        for caption in captions:
            tokens = self.tokenize_caption(caption)
            tokenized.append(tokens)

        tokens_tensor = torch.tensor(tokenized, device=next(self.parameters()).device)
        embeddings = self.embedding(tokens_tensor)
        embeddings += self.positional_encoding[:embeddings.size(1)].unsqueeze(0)

        text_features = self.transformer(embeddings)
        text_features = self.layer_norm(text_features.mean(dim=1))

        return text_features

In [ ]:
# ===================================================================
# EFFICIENT U-NET COMPONENTS
# ===================================================================

class EfficientResnetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim, text_emb_dim, dropout=0.1):
        super().__init__()
        groups = min(8, out_channels // 4) if out_channels >= 16 else 1

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        self.text_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(text_emb_dim, out_channels)
        )

        self.block1 = nn.Sequential(
            nn.GroupNorm(groups, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.Dropout2d(dropout)
        )

        self.block2 = nn.Sequential(
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.Dropout2d(dropout)
        )

        self.residual_conv = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, time_emb, text_emb):
        h = self.block1(x)

        time_emb = self.time_mlp(time_emb)
        h += time_emb[:, :, None, None]

        text_emb = self.text_mlp(text_emb)
        h += text_emb[:, :, None, None]

        h = self.block2(h)
        return h + self.residual_conv(x)

class OptimizedUNet(nn.Module):
    def __init__(self, in_channels=3, model_channels=Config.MODEL_CHANNELS, out_channels=3,
                 num_res_blocks=2, channel_mult=[1, 2, 3], num_heads=4,
                 text_emb_dim=Config.TEXT_EMB_DIM):
        super().__init__()

        self.in_channels = in_channels
        self.model_channels = model_channels
        self.out_channels = out_channels
        self.num_res_blocks = num_res_blocks
        self.channel_mult = channel_mult
        self.num_heads = num_heads

        time_embed_dim = model_channels * 3
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim),
        )

        self.input_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)

        # Build encoder
        self.encoder_blocks = nn.ModuleList()
        ch = model_channels
        input_block_chans = [model_channels]

        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [EfficientResnetBlock(ch, mult * model_channels, time_embed_dim, text_emb_dim)]
                ch = mult * model_channels
                self.encoder_blocks.append(nn.Sequential(*layers))
                input_block_chans.append(ch)

            if level != len(channel_mult) - 1:
                self.encoder_blocks.append(nn.Conv2d(ch, ch, 3, stride=2, padding=1))
                input_block_chans.append(ch)

        # Middle block
        self.middle_block = EfficientResnetBlock(ch, ch, time_embed_dim, text_emb_dim)

        # Build decoder
        self.decoder_blocks = nn.ModuleList()
        for level, mult in list(enumerate(channel_mult))[::-1]:
            for i in range(num_res_blocks + 1):
                ich = input_block_chans.pop()
                layers = [EfficientResnetBlock(ch + ich, model_channels * mult, time_embed_dim, text_emb_dim)]
                ch = model_channels * mult
                self.decoder_blocks.append(nn.Sequential(*layers))

            if level != 0:
                self.decoder_blocks.append(
                    nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)
                )

        # Output layers
        self.output_conv = nn.Sequential(
            nn.GroupNorm(min(8, model_channels // 4), model_channels),
            nn.SiLU(),
            nn.Conv2d(model_channels, out_channels, 3, padding=1)
        )

    def get_time_embedding(self, timesteps):
        half_dim = self.model_channels // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=timesteps.device) * -emb)
        emb = timesteps[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

    def forward(self, x, timesteps, text_emb):
        t_emb = self.get_time_embedding(timesteps)
        t_emb = self.time_embed(t_emb)

        h = self.input_conv(x)
        hs = [h]

        # Encoder
        for module in self.encoder_blocks:
            if isinstance(module, nn.Sequential):
                h = module[0](h, t_emb, text_emb)
            else:
                h = module(h)
            hs.append(h)

        # Middle
        h = self.middle_block(h, t_emb, text_emb)

        # Decoder
        for module in self.decoder_blocks:
            if isinstance(module, nn.Sequential):
                h = torch.cat([h, hs.pop()], dim=1)
                h = module[0](h, t_emb, text_emb)
            else:
                h = module(h)

        return self.output_conv(h)

In [ ]:
# ===================================================================
# DIFFUSION PROCESS
# ===================================================================

class SimpleDiffusion:
    def __init__(self, noise_steps=Config.NOISE_STEPS, beta_start=Config.BETA_START,
                 beta_end=Config.BETA_END, device='cuda'):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.device = device

        # Pre-compute diffusion parameters
        self.beta = torch.linspace(beta_start, beta_end, noise_steps).to(device)
        self.alpha = 1.0 - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def noise_images(self, x, t):
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        noise = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * noise, noise

    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.noise_steps, size=(n,), device=self.device)

    def sample(self, model, text_encoder, captions, n_samples=None, img_size=Config.IMAGE_SIZE):
        """Generate samples using the trained model"""
        if n_samples is None:
            n_samples = len(captions) if isinstance(captions, list) else 1

        if isinstance(captions, str):
            captions = [captions] * n_samples

        model.eval()
        with torch.no_grad():
            text_emb = text_encoder(captions)
            x = torch.randn((n_samples, 3, img_size, img_size)).to(self.device)

            for i in tqdm(reversed(range(1, self.noise_steps)), position=0, desc="Generating"):
                t = torch.full((n_samples,), i, dtype=torch.long, device=self.device)
                predicted_noise = model(x, t, text_emb)

                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]

                if i > 1:
                    noise = torch.randn_like(x)
                else:
                    noise = torch.zeros_like(x)

                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise

        model.train()
        return x


In [ ]:
# ===================================================================
# DATASET AND DATA LOADING
# ===================================================================

class YelpTrainingDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['photo_id']}.jpg")

        try:
            img = Image.open(img_path).convert("RGB")
            img_tensor = self.transform(img)
            caption = row["caption"]
            return img_tensor, caption
        except Exception as e:
            # Return dummy data if image loading fails
            dummy_img = torch.randn(3, Config.IMAGE_SIZE, Config.IMAGE_SIZE)
            return self.transform.transforms[-1](dummy_img), "a photo"

    def __len__(self):
        return len(self.df)

def create_better_caption(label):
    """Create more diverse and descriptive captions"""
    templates = {
        'food': [
            'a delicious plate of food at a restaurant',
            'tasty food served on a plate',
            'fresh restaurant meal beautifully presented',
            'appetizing dish from a restaurant kitchen'
        ],
        'menu': [
            'a restaurant menu showing food items',
            'menu board displaying meal options',
            'restaurant menu with various dishes listed',
            'dining menu with food choices'
        ],
        'inside': [
            'interior view of a restaurant or cafe',
            'cozy restaurant dining room interior',
            'modern restaurant interior design',
            'elegant cafe interior with tables and chairs'
        ],
        'outside': [
            'exterior view of a restaurant building',
            'restaurant building facade and entrance',
            'outdoor view of a dining establishment',
            'restaurant storefront and exterior'
        ],
        'drink': [
            'a refreshing drink or beverage',
            'cold beverage served at restaurant',
            'delicious drink in a glass',
            'refreshing beverage for dining'
        ]
    }

    label_templates = templates.get(label.lower(), [f"a photo of {label}"])
    return np.random.choice(label_templates)


In [ ]:
# ===================================================================
# EVALUATION METRICS
# ===================================================================

class ModelEvaluator:
    def __init__(self, device):
        self.device = device
        self.inception_model = inception_v3(pretrained=True, transform_input=False).eval().to(device)

    def compute_inception_features(self, images):
        """Compute inception features for FID calculation"""
        with torch.no_grad():
            images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
            features = self.inception_model(images)
            if isinstance(features, tuple):
                features = features[0]
            return features

    def compute_metrics(self, generated_images):
        """Compute Inception Score and simplified FID"""
        try:
            with torch.no_grad():
                # Ensure images are in correct range [0, 1]
                if generated_images.min() < 0:
                    generated_images = (generated_images + 1) / 2
                generated_images = torch.clamp(generated_images, 0, 1)

                # Inception Score
                resized_images = F.interpolate(generated_images, size=(299, 299), mode='bilinear', align_corners=False)
                preds = self.inception_model(resized_images)
                if isinstance(preds, tuple):
                    preds = preds[0]
                preds = F.softmax(preds, dim=1).cpu().numpy()

                # Calculate IS
                p_y = np.mean(preds, axis=0)
                kl_div = preds * (np.log(preds + 1e-10) - np.log(p_y + 1e-10))
                inception_score = np.exp(np.mean(np.sum(kl_div, axis=1)))

                # Simplified FID (just variance of generated features)
                gen_features = self.compute_inception_features(generated_images).cpu().numpy()
                mu_gen = np.mean(gen_features, axis=0)
                sigma_gen = np.cov(gen_features, rowvar=False)
                fid_score = np.trace(sigma_gen)

            return inception_score, fid_score

        except Exception as e:
            print(f"Error computing metrics: {e}")
            return 0.0, 0.0

In [ ]:
# ===================================================================
# TRAINING FUNCTIONS
# ===================================================================

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """Create learning rate scheduler with warmup"""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def train_model(unet, text_encoder, diffusion, dataloader, num_epochs, device):
    """Improved training function with all optimizations"""
    # Mixed precision training
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' and Config.USE_MIXED_PRECISION else None

    # Optimizers
    unet_optimizer = optim.AdamW(unet.parameters(), lr=Config.LEARNING_RATE,
                                weight_decay=Config.WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)
    text_optimizer = optim.AdamW(text_encoder.parameters(), lr=Config.LEARNING_RATE,
                                weight_decay=Config.WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)

    # Learning rate schedulers
    total_steps = len(dataloader) * num_epochs
    warmup_steps = total_steps // 10

    unet_scheduler = get_cosine_schedule_with_warmup(unet_optimizer, warmup_steps, total_steps)
    text_scheduler = get_cosine_schedule_with_warmup(text_optimizer, warmup_steps, total_steps)

    # EMA for better sample quality
    ema_unet = torch.optim.swa_utils.AveragedModel(unet, avg_fn=lambda avg, model, num:
                                                   Config.EMA_DECAY * avg + (1 - Config.EMA_DECAY) * model)

    unet.train()
    text_encoder.train()
    training_losses = []

    display_epochs = set([1] + list(range(5, num_epochs + 1, 5)))
    print("Starting improved model training...")

    for epoch in range(num_epochs):
        epoch_losses = []
        current_epoch = epoch + 1

        if current_epoch in display_epochs:
            pbar = tqdm(dataloader, desc=f"Epoch {current_epoch}/{num_epochs}")
        else:
            pbar = dataloader

        for batch_idx, batch_data in enumerate(pbar):
            try:
                if len(batch_data) != 2:
                    continue

                images, captions = batch_data
                if images is None or captions is None or len(images) == 0:
                    continue

                images = images.to(device)

                # Training step with optional mixed precision
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        text_emb = text_encoder(captions)
                        t = diffusion.sample_timesteps(images.shape[0])
                        x_t, noise = diffusion.noise_images(images, t)
                        predicted_noise = unet(x_t, t, text_emb)
                        loss = F.mse_loss(predicted_noise, noise)

                    unet_optimizer.zero_grad()
                    text_optimizer.zero_grad()
                    scaler.scale(loss).backward()
                    scaler.unscale_(unet_optimizer)
                    scaler.unscale_(text_optimizer)
                    torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=Config.GRADIENT_CLIP)
                    torch.nn.utils.clip_grad_norm_(text_encoder.parameters(), max_norm=Config.GRADIENT_CLIP)
                    scaler.step(unet_optimizer)
                    scaler.step(text_optimizer)
                    scaler.update()
                else:
                    text_emb = text_encoder(captions)
                    t = diffusion.sample_timesteps(images.shape[0])
                    x_t, noise = diffusion.noise_images(images, t)
                    predicted_noise = unet(x_t, t, text_emb)
                    loss = F.mse_loss(predicted_noise, noise)

                    unet_optimizer.zero_grad()
                    text_optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=Config.GRADIENT_CLIP)
                    torch.nn.utils.clip_grad_norm_(text_encoder.parameters(), max_norm=Config.GRADIENT_CLIP)
                    unet_optimizer.step()
                    text_optimizer.step()

                # Update EMA and schedulers
                ema_unet.update_parameters(unet)
                unet_scheduler.step()
                text_scheduler.step()

                epoch_losses.append(loss.item())

                if current_epoch in display_epochs and hasattr(pbar, 'set_postfix'):
                    pbar.set_postfix({'Loss': f'{loss.item():.4f}',
                                    'LR': f'{unet_optimizer.param_groups[0]["lr"]:.2e}'})

            except Exception as e:
                continue

        avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        training_losses.append(avg_loss)

        if current_epoch in display_epochs:
            print(f"Epoch {current_epoch}/{num_epochs} - Average Loss: {avg_loss:.4f}")

        # Save checkpoint
        if (epoch + 1) % Config.SAVE_INTERVAL == 0:
            config_dict = {k: v for k, v in Config.__dict__.items() if not k.startswith('__') and not callable(v)}
            checkpoint_path = f"improved_model_epoch_{epoch+1}.pt"
            torch.save({
                'unet_state_dict': unet.state_dict(),
                'ema_unet_state_dict': ema_unet.state_dict(),
                'text_encoder_state_dict': text_encoder.state_dict(),
                'unet_optimizer_state_dict': unet_optimizer.state_dict(),
                'text_optimizer_state_dict': text_optimizer.state_dict(),
                'epoch': epoch,
                'loss': avg_loss,
                'config': config_dict
            }, checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")

    return training_losses, ema_unet

In [ ]:
# ===================================================================
# MAIN MODEL CLASS
# ===================================================================

class GLIDELikeModel:
    def __init__(self, device='cuda'):
        self.device = device
        self.text_encoder = ImprovedTextEncoder().to(device)
        self.unet = OptimizedUNet().to(device)
        self.diffusion = SimpleDiffusion(device=device)
        self.evaluator = ModelEvaluator(device)
        self.ema_model = None

        # Enable optimizations for old systems
        if device == 'cuda':
            torch.backends.cudnn.benchmark = True

    def prepare_data(self, image_dir, metadata_file):
        """Prepare training data from Yelp dataset"""
        print("Preparing training data...")

        # Load metadata
        data = []
        with open(metadata_file, 'r') as f:
            for line in f:
                data.append(json.loads(line))
        df = pd.DataFrame(data)

        # Create better captions
        df['caption'] = df['label'].apply(create_better_caption)

        # Filter for available images
        available_images = [f.split('.')[0] for f in os.listdir(image_dir) if f.endswith('.jpg')]
        df_filtered = df[df['photo_id'].isin(available_images)].copy()

        # Balance dataset
        TARGET_LABELS = df_filtered["label"].unique().tolist()
        class_counts = df_filtered['label'].value_counts()
        min_samples = class_counts.min()

        samples = []
        for label in TARGET_LABELS:
            label_data = df_filtered[df_filtered["label"] == label]
            if len(label_data) >= min_samples:
                sampled_data = label_data.sample(min_samples, random_state=42)
            else:
                sampled_data = label_data
            samples.append(sampled_data)

        balanced_df = pd.concat(samples).reset_index(drop=True)

        print(f"Balanced dataset: {len(balanced_df)} samples")
        print("Class distribution:")
        for label in TARGET_LABELS:
            count = balanced_df['label'].value_counts().get(label, 0)
            print(f"  {label}: {count} samples")

        # Create dataset and dataloader
        transform = transforms.Compose([
            transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

        dataset = YelpTrainingDataset(balanced_df, image_dir, transform)
        dataloader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True if self.device == 'cuda' else False)

        return dataloader, balanced_df

    def train(self, dataloader, num_epochs=Config.NUM_EPOCHS):
        """Train the model"""
        print(f"Training model for {num_epochs} epochs...")
        losses, ema_model = train_model(
            self.unet, self.text_encoder, self.diffusion,
            dataloader, num_epochs, self.device
        )
        self.ema_model = ema_model
        return losses

    def generate_images(self, prompts, use_ema=True, save_path=None):
        """Generate images from text prompts"""
        model_to_use = self.ema_model if (use_ema and self.ema_model is not None) else self.unet

        if isinstance(prompts, str):
            prompts = [prompts]

        print(f"Generating {len(prompts)} images...")

        # Generate samples
        samples = self.diffusion.sample(
            model_to_use, self.text_encoder, prompts,
            n_samples=len(prompts), img_size=Config.IMAGE_SIZE
        )

        # Denormalize images
        samples = torch.clamp((samples + 1) / 2, 0, 1)

        if save_path:
            # Save images
            for i, (sample, prompt) in enumerate(zip(samples, prompts)):
                filename = f"{save_path}_prompt_{i}_{prompt[:20].replace(' ', '_')}.png"
                save_image(sample, filename)
            print(f"Images saved with prefix: {save_path}")

        return samples

    def evaluate_samples(self, samples):
        """Evaluate generated samples using IS and FID"""
        print("Evaluating samples...")
        is_score, fid_score = self.evaluator.compute_metrics(samples)

        print(f"Inception Score: {is_score:.4f}")
        print(f"FID Score: {fid_score:.4f}")

        return is_score, fid_score

    def visualize_samples(self, samples, prompts, save_path=None):
        """Visualize generated samples"""
        if isinstance(prompts, str):
            prompts = [prompts]

        n_samples = len(samples)
        cols = min(5, n_samples)
        rows = (n_samples + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1 or cols == 1:
            axes = axes.flatten()
        else:
            axes = axes.flatten()

        for i in range(n_samples):
            img = samples[i].cpu().permute(1, 2, 0).numpy()
            axes[i].imshow(img)
            axes[i].set_title(f"'{prompts[i][:30]}{'...' if len(prompts[i]) > 30 else ''}",
                            fontsize=10, wrap=True)
            axes[i].axis('off')

        # Hide unused subplots
        for i in range(n_samples, len(axes)):
            axes[i].axis('off')

        plt.tight_layout()
        if save_path:
            plt.savefig(f"{save_path}_visualization.png", dpi=300, bbox_inches='tight')
            print(f"Visualization saved: {save_path}_visualization.png")
        plt.show()

    def save_model(self, path):
        """Save the trained model"""
        # Extract only the serializable parts of the config
        config_dict = {
            k: v for k, v in Config.__dict__.items()
            if not k.startswith('__') and not callable(v)
        }

        checkpoint = {
            'unet_state_dict': self.unet.state_dict(),
            'text_encoder_state_dict': self.text_encoder.state_dict(),
            'config': config_dict
        }

        if self.ema_model is not None:
            checkpoint['ema_unet_state_dict'] = self.ema_model.state_dict()

        torch.save(checkpoint, path)
        print(f"Model saved to: {path}")


    def load_model(self, path):
        """Load a trained model"""
        checkpoint = torch.load(path, map_location=self.device)

        self.unet.load_state_dict(checkpoint['unet_state_dict'])
        self.text_encoder.load_state_dict(checkpoint['text_encoder_state_dict'])

        if 'ema_unet_state_dict' in checkpoint:
            self.ema_model = torch.optim.swa_utils.AveragedModel(self.unet)
            self.ema_model.load_state_dict(checkpoint['ema_unet_state_dict'])

        print(f"Model loaded from: {path}")



In [ ]:
# ===================================================================
# MAIN PIPELINE FUNCTION
# ===================================================================

def main_pipeline(image_dir="./balanced_photos_folder", metadata_file="./photos.json",
                 train_from_scratch=True, model_path=None):
    """
    Main pipeline to train and use the GLIDE-like model

    Args:
        image_dir: Path to directory containing training images
        metadata_file: Path to JSON file with image metadata
        train_from_scratch: Whether to train a new model or load existing
        model_path: Path to saved model (for loading) or where to save trained model
    """

    print("="*80)
    print("GLIDE-LIKE MODEL PIPELINE")
    print("="*80)

    # Setup device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Initialize model
    model = GLIDELikeModel(device=device)

    if train_from_scratch:
        print("\n" + "="*50)
        print("TRAINING MODE")
        print("="*50)

        # Prepare data
        dataloader, df_train = model.prepare_data(image_dir, metadata_file)

        # Train model
        losses = model.train(dataloader, num_epochs=Config.NUM_EPOCHS)

        # Plot training loss
        plt.figure(figsize=(10, 6))
        plt.plot(losses, linewidth=2, color='blue')
        plt.title('Training Loss Over Time', fontsize=16)
        plt.xlabel('Epoch', fontsize=14)
        plt.ylabel('Loss (MSE)', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Save trained model
        if model_path is None:
            model_path = "glide_like_model_final.pt"
        model.save_model(model_path)

    else:
        print("\n" + "="*50)
        print("LOADING EXISTING MODEL")
        print("="*50)

        if model_path is None:
            raise ValueError("model_path must be provided when train_from_scratch=False")

        model.load_model(model_path)

    return model

def demo_generation(model, custom_prompts=None):
    """
    Demonstrate image generation with the trained model

    Args:
        model: Trained GLIDELikeModel instance
        custom_prompts: List of custom prompts, or None for default prompts
    """

    print("\n" + "="*50)
    print("DEMONSTRATION: IMAGE GENERATION")
    print("="*50)

    # Default prompts if none provided
    if custom_prompts is None:
        demo_prompts = [
            "a delicious plate of food at a restaurant",
            "interior view of a restaurant or cafe",
            "a restaurant menu showing food items",
            "exterior view of a restaurant building",
            "a refreshing drink or beverage"
        ]
    else:
        demo_prompts = custom_prompts

    print(f"Generating images for {len(demo_prompts)} prompts:")
    for i, prompt in enumerate(demo_prompts, 1):
        print(f"  {i}. '{prompt}'")

    # Generate images
    samples = model.generate_images(demo_prompts, use_ema=True, save_path="demo_output")

    # Evaluate samples
    is_score, fid_score = model.evaluate_samples(samples)

    # Visualize results
    model.visualize_samples(samples, demo_prompts, save_path="demo_results")

    print(f"\nGeneration completed!")
    print(f"Inception Score: {is_score:.4f}")
    print(f"FID Score: {fid_score:.4f}")

    return samples, is_score, fid_score

def interactive_generation(model):
    """
    Interactive mode for generating images from custom prompts
    """

    print("\n" + "="*50)
    print("INTERACTIVE GENERATION MODE")
    print("="*50)
    print("Enter your custom prompts (type 'quit' to exit):")

    while True:
        prompt = input("\nEnter prompt: ").strip()

        if prompt.lower() in ['quit', 'exit', 'q']:
            print("Exiting interactive mode...")
            break

        if not prompt:
            print("Please enter a valid prompt!")
            continue

        try:
            print(f"Generating image for: '{prompt}'")

            # Generate single image
            samples = model.generate_images([prompt], use_ema=True,
                                          save_path=f"interactive_{int(time.time())}")

            # Quick evaluation
            is_score, fid_score = model.evaluate_samples(samples)

            # Show result
            model.visualize_samples(samples, [prompt])

            print(f"IS: {is_score:.4f}, FID: {fid_score:.4f}")

        except Exception as e:
            print(f"Error generating image: {e}")
            continue


In [ ]:
# ===================================================================
# TEXT INPUT FUNCTIONS
# ===================================================================

def get_user_text_input():
    """Get text input from user for image generation"""
    print("\n" + "="*60)
    print("TEXT-TO-IMAGE GENERATION")
    print("="*60)
    print("Enter your text descriptions for image generation:")
    print("(You can enter multiple prompts, press Enter after each)")
    print("(Type 'done' when finished, or 'quit' to exit)")
    print("-"*60)

    prompts = []
    while True:
        prompt = input(f"Prompt {len(prompts)+1}: ").strip()

        if prompt.lower() in ['quit', 'exit', 'q']:
            if len(prompts) == 0:
                print("No prompts entered. Exiting...")
                return None
            break
        elif prompt.lower() in ['done', 'finish', 'd']:
            if len(prompts) == 0:
                print("No prompts entered. Please enter at least one prompt.")
                continue
            break
        elif prompt == "":
            print("Empty prompt. Please enter a description or type 'done' to finish.")
            continue
        else:
            prompts.append(prompt)
            print(f"✓ Added: '{prompt}'")

    print(f"\n📝 You entered {len(prompts)} prompt(s):")
    for i, p in enumerate(prompts, 1):
        print(f"  {i}. {p}")

    return prompts

def text_to_image_pipeline(model_path="glide_like_model_final.pt"):
    """Complete text-to-image pipeline with user input"""
    print("🎨 GLIDE-LIKE TEXT-TO-IMAGE GENERATOR")
    print("="*60)

    # Setup device and load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    try:
        model = GLIDELikeModel(device=device)
        print(f"Loading model from: {model_path}")
        model.load_model(model_path)
        print("✅ Model loaded successfully!")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Please make sure you have a trained model or train one first.")
        return None

    # Get user input
    prompts = get_user_text_input()
    if prompts is None:
        return None

    # Generate images
    print(f"\n🎯 Generating {len(prompts)} image(s)...")
    try:
        samples = model.generate_images(
            prompts,
            use_ema=True,
            save_path=f"user_generated_{int(time.time())}"
        )

        # Evaluate samples
        print("\n📊 Evaluating generated images...")
        is_score, fid_score = model.evaluate_samples(samples)

        # Visualize results
        print("\n🖼️  Displaying results...")
        model.visualize_samples(samples, prompts, save_path=f"user_results_{int(time.time())}")

        print(f"\n✅ Generation completed successfully!")
        print(f"📈 Inception Score: {is_score:.4f}")
        print(f"📉 FID Score: {fid_score:.4f}")

        return samples, is_score, fid_score

    except Exception as e:
        print(f"❌ Error during generation: {e}")
        return None

def simple_text_input(text_prompt, model_path="glide_like_model_final.pt"):
    """Simple function to generate image from a single text prompt"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    try:
        model = GLIDELikeModel(device=device)
        model.load_model(model_path)

        print(f"🎨 Generating image for: '{text_prompt}'")
        samples = model.generate_images([text_prompt], save_path="simple_gen")
        model.visualize_samples(samples, [text_prompt])

        return samples[0]  # Return single image

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def command_line_interface():
    """Command line interface for the model"""
    import argparse

    parser = argparse.ArgumentParser(description='GLIDE-like Text-to-Image Generator')
    parser.add_argument('--mode', choices=['train', 'generate', 'interactive'],
                       default='interactive', help='Mode to run the model in')
    parser.add_argument('--prompt', type=str, help='Text prompt for image generation')
    parser.add_argument('--model_path', type=str, default='glide_like_model_final.pt',
                       help='Path to saved model')
    parser.add_argument('--image_dir', type=str, default='./balanced_photos_folder',
                       help='Directory containing training images')
    parser.add_argument('--metadata_file', type=str, default='./photos.json',
                       help='JSON file with image metadata')
    parser.add_argument('--epochs', type=int, default=35, help='Number of training epochs')

    args = parser.parse_args()

    if args.mode == 'train':
        print("🚀 Starting training mode...")
        model = main_pipeline(
            image_dir=args.image_dir,
            metadata_file=args.metadata_file,
            train_from_scratch=True,
            model_path=args.model_path
        )
        print("✅ Training completed!")

    elif args.mode == 'generate':
        if args.prompt:
            print(f"🎨 Generating image for: '{args.prompt}'")
            simple_text_input(args.prompt, args.model_path)
        else:
            print("❌ Please provide a --prompt for generation mode")

    elif args.mode == 'interactive':
        text_to_image_pipeline(args.model_path)


In [ ]:
# ===================================================================
# EXAMPLE USAGE SCRIPTS
# ===================================================================
"""
if __name__ == "__main__":
    # Choose what to run:

    # Option 1: Interactive text-to-image (DEFAULT)
    print("🎨 Welcome to GLIDE-like Text-to-Image Generator!")
    print("Choose an option:")
    print("1. Interactive text-to-image generation")
    print("2. Train new model")
    print("3. Quick demo with sample prompts")

    choice = input("Enter your choice (1/2/3): ").strip()

    if choice == "1" or choice == "":
        # Interactive mode - get text input from user
        text_to_image_pipeline()

    elif choice == "2":
        # Training mode
        print("🚀 Starting training pipeline...")
        model = main_pipeline(
            image_dir="/content/drive/MyDrive/balanced_photos_folder",
            metadata_file="/content/photos.json",
            train_from_scratch=True,
            model_path="/content/glide_like_model_final.pt"
        )
        print("✅ Training completed! You can now generate images.")

        # Ask if user wants to generate images
        gen_choice = input("Would you like to generate some images now? (y/n): ").strip().lower()
        if gen_choice == 'y' or gen_choice == 'yes':
            text_to_image_pipeline()

    elif choice == "3":
        # Demo mode
        try:
            model = main_pipeline(train_from_scratch=False, model_path="glide_like_model_final.pt")
            demo_generation(model)
        except:
            print("❌ No trained model found. Please train a model first (option 2).")

    else:
        print("Invalid choice. Running interactive mode by default...")
        text_to_image_pipeline()
"""

In [ ]:
# ===================================================================
# QUICK START FUNCTIONS
# ===================================================================

def quick_train():
    """Quick training function - just run this to start training"""
    model = main_pipeline(train_from_scratch=True)
    return model

def quick_generate(prompt, model_path="glide_like_model_final.pt"):
    """Quick generation function - generate image from prompt"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GLIDELikeModel(device=device)
    model.load_model(model_path)

    samples = model.generate_images([prompt], save_path="quick_gen")
    model.visualize_samples(samples, [prompt])

    return samples

def batch_generate(prompts, model_path="glide_like_model_final.pt"):
    """Generate multiple images at once"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GLIDELikeModel(device=device)
    model.load_model(model_path)

    samples = model.generate_images(prompts, save_path="batch_gen")
    is_score, fid_score = model.evaluate_samples(samples)
    model.visualize_samples(samples, prompts)

    print(f"Batch generation completed - IS: {is_score:.4f}, FID: {fid_score:.4f}")
    return samples, is_score, fid_score


In [ ]:
# ===================================================================
# CATEGORY SAMPLE GENERATION BLOCK
# ===================================================================

def generate_category_samples(model, samples_per_category=3, save_individual=True,
                            model_path="glide_like_model_final.pt"):
    """
    Generate sample images for each category in the dataset

    Args:
        model: Trained GLIDELikeModel instance (optional, will load if None)
        samples_per_category: Number of sample images to generate per category
        save_individual: Whether to save individual images for each sample
        model_path: Path to saved model (if model is None)

    Returns:
        dict: Dictionary containing generated samples organized by category
    """

    print("\n" + "="*70)
    print("🎨 GENERATING CATEGORY SAMPLES")
    print("="*70)

    # Load model if not provided
    if model is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = GLIDELikeModel(device=device)
        try:
            print(f"Loading model from: {model_path}")
            model.load_model(model_path)
            print("✅ Model loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return None

    # Define categories and their variations
    categories = {
        'food': [
            'a delicious plate of food at a restaurant',
            'tasty food served on a plate',
            'fresh restaurant meal beautifully presented',
            'appetizing dish from a restaurant kitchen',
            'gourmet food plated elegantly',
            'colorful fresh meal at dining table'
        ],
        'menu': [
            'a restaurant menu showing food items',
            'menu board displaying meal options',
            'restaurant menu with various dishes listed',
            'dining menu with food choices',
            'elegant restaurant menu card',
            'modern digital menu display'
        ],
        'inside': [
            'interior view of a restaurant or cafe',
            'cozy restaurant dining room interior',
            'modern restaurant interior design',
            'elegant cafe interior with tables and chairs',
            'warm restaurant atmosphere with lighting',
            'bustling restaurant dining area'
        ],
        'outside': [
            'exterior view of a restaurant building',
            'restaurant building facade and entrance',
            'outdoor view of a dining establishment',
            'restaurant storefront and exterior',
            'charming cafe exterior with windows',
            'modern restaurant building design'
        ],
        'drink': [
            'a refreshing drink or beverage',
            'cold beverage served at restaurant',
            'delicious drink in a glass',
            'refreshing beverage for dining',
            'colorful cocktail with garnish',
            'steaming hot coffee in elegant cup'
        ]
    }

    all_samples = {}
    all_prompts = {}
    timestamp = int(time.time())

    print(f"Generating {samples_per_category} samples for each of {len(categories)} categories...")
    print("-" * 70)

    for category, prompt_variations in categories.items():
        print(f"\n🏷️  Generating samples for category: {category.upper()}")

        # Select prompts for this category
        selected_prompts = np.random.choice(
            prompt_variations,
            size=min(samples_per_category, len(prompt_variations)),
            replace=False
        ).tolist()

        # If we need more samples than available prompts, cycle through them
        if samples_per_category > len(prompt_variations):
            remaining = samples_per_category - len(prompt_variations)
            additional_prompts = np.random.choice(prompt_variations, size=remaining, replace=True).tolist()
            selected_prompts.extend(additional_prompts)

        # Display selected prompts
        for i, prompt in enumerate(selected_prompts, 1):
            print(f"  {i}. '{prompt}'")

        try:
            # Generate images for this category
            print(f"  🎯 Generating {len(selected_prompts)} images...")

            samples = model.generate_images(
                selected_prompts,
                use_ema=True,
                save_path=f"category_{category}_{timestamp}" if save_individual else None
            )

            # Store results
            all_samples[category] = samples
            all_prompts[category] = selected_prompts

            print(f"  ✅ Generated {len(samples)} images for {category}")

        except Exception as e:
            print(f"  ❌ Error generating samples for {category}: {e}")
            all_samples[category] = None
            all_prompts[category] = selected_prompts
            continue

    # Create comprehensive visualization
    print(f"\n📊 Creating comprehensive visualization...")
    create_category_grid_visualization(all_samples, all_prompts, timestamp)

    # Generate evaluation report
    print(f"\n📈 Evaluating generated samples...")
    evaluation_results = evaluate_category_samples(model, all_samples, all_prompts)

    # Print summary
    print_category_generation_summary(all_samples, evaluation_results)

    return {
        'samples': all_samples,
        'prompts': all_prompts,
        'evaluation': evaluation_results,
        'timestamp': timestamp
    }

def create_category_grid_visualization(all_samples, all_prompts, timestamp):
    """Create a comprehensive grid visualization of all category samples"""

    # Count valid categories
    valid_categories = [cat for cat, samples in all_samples.items() if samples is not None]

    if not valid_categories:
        print("❌ No valid samples to visualize")
        return

    # Calculate grid dimensions
    n_categories = len(valid_categories)
    samples_per_cat = len(all_samples[valid_categories[0]]) if valid_categories else 0

    # Create figure
    fig_width = min(20, samples_per_cat * 4)
    fig_height = min(24, n_categories * 4)
    fig, axes = plt.subplots(n_categories, samples_per_cat,
                            figsize=(fig_width, fig_height))

    # Handle single row or column cases
    if n_categories == 1:
        axes = axes.reshape(1, -1)
    elif samples_per_cat == 1:
        axes = axes.reshape(-1, 1)

    fig.suptitle('Generated Samples by Category', fontsize=20, y=0.98)

    for cat_idx, category in enumerate(valid_categories):
        samples = all_samples[category]
        prompts = all_prompts[category]

        for sample_idx in range(samples_per_cat):
            ax = axes[cat_idx, sample_idx] if n_categories > 1 else axes[sample_idx]

            if sample_idx < len(samples):
                # Display image
                img = samples[sample_idx].cpu().permute(1, 2, 0).numpy()
                img = np.clip(img, 0, 1)  # Ensure valid range
                ax.imshow(img)

                # Set title
                prompt_short = prompts[sample_idx][:40] + "..." if len(prompts[sample_idx]) > 40 else prompts[sample_idx]
                ax.set_title(f"{category.upper()}\n'{prompt_short}'",
                           fontsize=10, pad=10)
            else:
                ax.text(0.5, 0.5, 'No Image', ha='center', va='center',
                       transform=ax.transAxes, fontsize=12)
                ax.set_title(f"{category.upper()}\nEmpty", fontsize=10)

            ax.axis('off')

    plt.tight_layout()

    # Save the comprehensive visualization
    save_path = f"category_samples_grid_{timestamp}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"📁 Comprehensive grid saved: {save_path}")
    plt.show()

def evaluate_category_samples(model, all_samples, all_prompts):
    """Evaluate samples for each category"""

    evaluation_results = {}

    for category, samples in all_samples.items():
        if samples is None:
            evaluation_results[category] = {'is_score': 0.0, 'fid_score': 0.0, 'error': True}
            continue

        try:
            print(f"  📊 Evaluating {category} samples...")
            is_score, fid_score = model.evaluate_samples(samples)
            evaluation_results[category] = {
                'is_score': is_score,
                'fid_score': fid_score,
                'error': False,
                'n_samples': len(samples)
            }
        except Exception as e:
            print(f"  ❌ Error evaluating {category}: {e}")
            evaluation_results[category] = {'is_score': 0.0, 'fid_score': 0.0, 'error': True}

    return evaluation_results

def print_category_generation_summary(all_samples, evaluation_results):
    """Print a comprehensive summary of the category generation"""

    print("\n" + "="*70)
    print("📋 CATEGORY GENERATION SUMMARY")
    print("="*70)

    total_samples = 0
    successful_categories = 0

    print(f"{'Category':<12} {'Samples':<8} {'IS Score':<10} {'FID Score':<12} {'Status':<10}")
    print("-" * 70)

    for category in ['food', 'menu', 'inside', 'outside', 'drink']:
        if category in all_samples:
            samples = all_samples[category]
            eval_data = evaluation_results.get(category, {})

            if samples is not None and not eval_data.get('error', True):
                n_samples = len(samples)
                is_score = eval_data.get('is_score', 0.0)
                fid_score = eval_data.get('fid_score', 0.0)
                status = "✅ Success"
                total_samples += n_samples
                successful_categories += 1

                print(f"{category:<12} {n_samples:<8} {is_score:<10.4f} {fid_score:<12.4f} {status:<10}")
            else:
                print(f"{category:<12} {'0':<8} {'N/A':<10} {'N/A':<12} {'❌ Failed':<10}")
        else:
            print(f"{category:<12} {'0':<8} {'N/A':<10} {'N/A':<12} {'⚠️ Skipped':<10}")

    print("-" * 70)
    print(f"📊 OVERALL STATISTICS:")
    print(f"   • Total samples generated: {total_samples}")
    print(f"   • Successful categories: {successful_categories}/5")
    print(f"   • Success rate: {(successful_categories/5)*100:.1f}%")

    if successful_categories > 0:
        avg_is = np.mean([eval_data['is_score'] for eval_data in evaluation_results.values()
                         if not eval_data.get('error', True)])
        avg_fid = np.mean([eval_data['fid_score'] for eval_data in evaluation_results.values()
                          if not eval_data.get('error', True)])
        print(f"   • Average IS Score: {avg_is:.4f}")
        print(f"   • Average FID Score: {avg_fid:.4f}")

def quick_category_samples(samples_per_category=2, model_path="glide_like_model_final.pt"):
    """Quick function to generate category samples with minimal setup"""

    print("🚀 Quick Category Sample Generation")
    return generate_category_samples(
        model=None,
        samples_per_category=samples_per_category,
        model_path=model_path
    )

def category_comparison_analysis(model, model_path="glide_like_model_final.pt"):
    """Generate samples and create detailed comparison analysis"""

    print("\n" + "="*70)
    print("🔍 DETAILED CATEGORY COMPARISON ANALYSIS")
    print("="*70)

    # Generate samples for analysis
    results = generate_category_samples(model, samples_per_category=5, model_path=model_path)

    if results is None:
        return None

    # Create individual category visualizations
    for category, samples in results['samples'].items():
        if samples is not None:
            prompts = results['prompts'][category]

            # Create individual category visualization
            plt.figure(figsize=(15, 3))
            for i, (sample, prompt) in enumerate(zip(samples, prompts)):
                plt.subplot(1, len(samples), i+1)
                img = sample.cpu().permute(1, 2, 0).numpy()
                img = np.clip(img, 0, 1)
                plt.imshow(img)
                plt.title(f"'{prompt[:25]}{'...' if len(prompt) > 25 else ''}",
                         fontsize=8, wrap=True)
                plt.axis('off')

            plt.suptitle(f'{category.upper()} Category Samples', fontsize=14, y=1.02)
            plt.tight_layout()

            # Save individual category visualization
            save_path = f"category_{category}_detailed_{results['timestamp']}.png"
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"📁 {category} detailed view saved: {save_path}")
            plt.show()

    return results


In [ ]:
# ===================================================================
# INTEGRATION WITH MAIN PIPELINE
# ===================================================================

def enhanced_demo_with_categories(model=None, model_path="glide_like_model_final.pt"):
    """Enhanced demo that includes category sample generation"""

    print("\n" + "="*70)
    print("🎨 ENHANCED DEMO WITH CATEGORY SAMPLES")
    print("="*70)

    # Load model if not provided
    if model is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = GLIDELikeModel(device=device)
        try:
            model.load_model(model_path)
            print("✅ Model loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            return None

    # Generate category samples
    category_results = generate_category_samples(model, samples_per_category=3)

    if category_results is None:
        print("❌ Category sample generation failed")
        return None

    # Run additional analysis
    print("\n🔍 Running additional analysis...")

    # Calculate diversity metrics
    all_valid_samples = []
    for samples in category_results['samples'].values():
        if samples is not None:
            all_valid_samples.extend(samples)

    if all_valid_samples:
        print(f"📊 Generated {len(all_valid_samples)} total samples across all categories")

        # Overall evaluation
        overall_is, overall_fid = model.evaluate_samples(torch.stack(all_valid_samples))
        print(f"🎯 Overall IS Score: {overall_is:.4f}")
        print(f"🎯 Overall FID Score: {overall_fid:.4f}")

    return category_results

# Add this to the main execution block at the end of the file
def run_category_sample_demo(samples_per_category=3):
    """Run the category sample generation demo"""
    print("\n" + "🎨" * 25)
    print("CATEGORY SAMPLE GENERATION DEMO")
    print("🎨" * 25)

    try:
        # Try to load existing model and generate category samples
        results = quick_category_samples(samples_per_category=samples_per_category)

        if results:
            print("\n✅ Category sample generation completed successfully!")
            print("📁 Check the generated files for detailed visualizations.")
            return results
        else:
            print("\n❌ Category sample generation failed.")
            print("💡 Make sure you have a trained model available.")
            return None

    except Exception as e:
        print(f"\n❌ Error during category sample generation: {e}")
        print("💡 Try training a model first using option 2 in the main menu.")
        return None


In [ ]:
# ===================================================================
# UPDATED MAIN EXECUTION BLOCK
# ===================================================================

# Replace the existing main execution block with this enhanced version:

if __name__ == "__main__":
    # Enhanced main menu with category sample generation
    print("🎨 Welcome to GLIDE-like Text-to-Image Generator!")
    print("Choose an option:")
    print("1. Interactive text-to-image generation")
    print("2. Train new model")
    print("3. Quick demo with sample prompts")
    print("4. Generate samples for each category")  # NEW OPTION
    print("5. Detailed category analysis")          # NEW OPTION

    choice = input("Enter your choice (1/2/3/4/5): ").strip()

    if choice == "1" or choice == "":
        # Interactive mode - get text input from user
        text_to_image_pipeline()

    elif choice == "2":
        # Training mode
        print("🚀 Starting training pipeline...")
        model = main_pipeline(
            image_dir="/content/drive/MyDrive/balanced_photos_folder",
            metadata_file="/content/drive/MyDrive/photos.json",
            train_from_scratch=True,
            model_path="glide_like_model_final.pt"
        )
        print("✅ Training completed! You can now generate images.")

        # Ask if user wants to generate category samples
        gen_choice = input("Would you like to generate category samples now? (y/n): ").strip().lower()
        if gen_choice == 'y' or gen_choice == 'yes':
            run_category_sample_demo()

    elif choice == "3":
        # Demo mode
        try:
            model = main_pipeline(train_from_scratch=False, model_path="glide_like_model_final.pt")
            demo_generation(model)
        except:
            print("❌ No trained model found. Please train a model first (option 2).")

    elif choice == "4":
        # NEW: Category sample generation
        run_category_sample_demo()

    elif choice == "5":
        # NEW: Detailed category analysis
        try:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = GLIDELikeModel(device=device)
            model.load_model("glide_like_model_final.pt")
            category_comparison_analysis(model)
        except Exception as e:
            print(f"❌ Error: {e}")
            print("💡 Make sure you have a trained model available.")

    else:
        print("Invalid choice. Running interactive mode by default...")
        text_to_image_pipeline()


In [ ]:
try:
  model = main_pipeline(train_from_scratch=False, model_path="/content/glide_like_model_final.pt")
  demo_generation(model)
except:
  print("❌ No trained model found. Please train a model first (option 2).")


In [ ]:
text_to_image_pipeline()

GLIDE-LIKE vs ORIGINAL GLIDE MODEL COMPARISON
============================================================

DATASET INFORMATION:
  - Total samples used: 8390
  - Labels: inside, outside, drink, food, menu
  - Balanced sampling based on smallest class
  - Image size: 64x64

TRAINING CONFIGURATION:
  - Epochs: 50
  - Batch Size: 20
  - Learning Rate: 2e-05
  - Optimizer: AdamW with Cosine Annealing

GLIDE-LIKE MODEL (Built from Scratch):
  - Architecture: Custom U-Net with text conditioning
  - Text Encoder: Simple transformer-based encoder
  - Diffusion: DDPM-style process
  - Inception Score: 1.9394
  - FID Score: 824.6035
  - Training Loss: 0.0226

ORIGINAL GLIDE MODEL (Pre-trained):
  - Architecture: OpenAI's GLIDE implementation
  - Text Encoder: CLIP-based encoder
  - Diffusion: Advanced guided diffusion
  - Inception Score: 3.4624
  - FID Score: 1007.9089
  - Pre-trained: Yes (on large-scale dataset)

COMPARISON SUMMARY:
  - IS Score Comparison: Original GLIDE (3.4624 vs 1.9394)
  - FID Score Comparison: GLIDE-like (lower is better: 824.6035 vs 1007.9089)

